# Chapter 7: Multiple Regression Analysis with Qualitative Regressors

Topics: dummy variables, boolean variables, categorical variables, ANOVA, interaction with dummies.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('wage1')

## 7.1 Dummy Variables as Regressors

In [ ]:
# Gender wage gap
model = smf.ols('wage ~ educ + exper + tenure + female', data=df).fit()
print(model.summary())
print(f"\nGender gap: women earn ${abs(model.params['female']):.2f}/hr less, ceteris paribus")

## 7.2 Boolean Variables

In [ ]:
df['married'] = df['married'].astype(bool)
df['female']  = df['female'].astype(bool)

m_bool = smf.ols('wage ~ educ + exper + tenure + married + female', data=df).fit()
print(m_bool.summary())

## 7.3 Categorical Variables and ANOVA

In [ ]:
# Use number of dependents as categorical
m_cat = smf.ols('wage ~ educ + exper + C(numdep)', data=df).fit()
print(m_cat.summary())

# ANOVA
anova_table = sm.stats.anova_lm(m_cat, typ=2)
print("\nANOVA Table:")
print(anova_table)

## 7.5 Interactions and Differences Across Groups

In [ ]:
# Chow test approach: separate slopes by gender
m_gender = smf.ols('wage ~ educ * female + exper * female + tenure', data=df).fit()
print(m_gender.summary())

# Plot wage-education line by gender
educ_range = np.linspace(0, 18, 100)
b = m_gender.params
wage_men   = b['Intercept'] + b['educ'] * educ_range
wage_women = (b['Intercept'] + b['female']) + (b['educ'] + b['educ:female']) * educ_range

plt.figure(figsize=(8, 5))
plt.plot(educ_range, wage_men,   label='Men',   color='steelblue', lw=2)
plt.plot(educ_range, wage_women, label='Women', color='darkorange', lw=2)
plt.xlabel('Education'); plt.ylabel('Predicted Wage')
plt.title('Wage-Education Relationship by Gender')
plt.legend()
plt.savefig(f"{output_dir}/fig1_gender_wage_educ.png", dpi=150)
plt.show()